# Dataset、DataLoader 与数据边界

## 学习目标

能够构建 Dataset、划分训练/验证/测试索引，并解释增强、shuffle 和 batch 的边界。


## 概念模型与执行路径

Dataset 定义如何取得单个样本，DataLoader 定义如何采样、组批和并行加载。训练和验证可以来自同一原始数据，但必须使用独立 transform；测试集不能参与模型选择。


### 实验 1


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split

features = torch.arange(40, dtype=torch.float32).reshape(20, 2)
labels = (features.sum(dim=1) > 30).long()
dataset = TensorDataset(features, labels)
train_set, validation_set = random_split(dataset, [16, 4], generator=torch.Generator().manual_seed(42))
print(len(train_set), len(validation_set), set(train_set.indices).isdisjoint(validation_set.indices))


### 实验 2


In [ ]:
loader = DataLoader(train_set, batch_size=6, shuffle=True)
for batch_index, (inputs, targets) in enumerate(loader):
    print(batch_index, inputs.shape, targets.shape)


### 实验 3


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 4


In [ ]:
# 该单元首次运行会下载真实 MNIST。
from common.data import image_loaders
train_loader, validation_loader, test_loader, channels = image_loaders(
    "mnist", PYTORCH_ROOT / "data", batch_size=64, quick=True, augment=True
)
images, labels = next(iter(train_loader))
print("MNIST batch:", images.shape, labels.shape, "channels:", channels)
print("split sizes:", len(train_loader.dataset), len(validation_loader.dataset), len(test_loader.dataset))


## 底层机制

随机增强属于 Dataset transform，因此若训练和验证共享同一个 Dataset 对象，验证也会被增强。课程实现使用相同索引、不同 Dataset 实例来隔离 transform。


## 检查点

为什么训练集需要 shuffle，而验证集通常不需要？这是否影响指标数学结果？


## 试一试

把 batch size 改为 7，预测最后一个 batch 的大小；为 TensorDataset 写一个自定义 collate_fn。


## 常见错误与调试

划分后再对共享 Dataset 修改 transform、对时间序列随机打乱造成泄漏、验证集使用训练增强、`num_workers` 过大拖慢小数据集。
